In [17]:
import pandas as pd
import numpy as np
import datetime

import warnings
warnings.filterwarnings('ignore')

In [63]:
# Преобразование индексов в виду datetime для каждого массива данных
def set_datetime_index_ace(df):
    date_values = ['year', 'month', 'day', 'hour from']
    date = df[date_values].copy()

    date['hour from'] = date['hour from'].apply(lambda x: f'{x:02d}')
    date['datetime'] = pd.to_datetime(
        date['year'].astype(str) + '-' +
        date['month'].astype(str) + '-' +
        date['day'].astype(str) + ' ' +
        date['hour from']
    )
    
    dataset = df.copy()
    dataset = dataset.set_index(date['datetime'])
    dataset = dataset.drop(columns=date_values) 
    return dataset

def set_datetime_index_discover(df, date_column='Date time'):
    df_copy = df.copy()
    # df_copy['datetime'] = pd.to_datetime(df_copy[date_column], format='%d.%m.%Y %H:%M')
    df_copy['datetime'] = pd.to_datetime(df_copy[date_column], format='ISO8601').dt.floor('S') #format="%Y-%m-%d %H:%M:%S")
    df_copy = df_copy.set_index('datetime')
    df_copy = df_copy.drop(columns=[date_column])  
    return df_copy

def set_cos_sin(df):
    df_copy = df.copy()
    dt_index = pd.to_datetime(df_copy.index)

    # день года как циклическая фича (1–365/366)
    day_of_year = dt_index.dayofyear
    df_copy['doy_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
    df_copy['doy_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)

    return df_copy
    
# Осуществление погружения временного ряда (delay embedding):
def create_lagged_features(df, vars_list, embed_map_or_int, prediction_horizon=3):
    """
    df: массив данных - датафрейм с индексом 'datetime'
    vars_list: список имён переменных, которые погружаем
    embed_map_or_int: заданные глубины погружения - либо int (одинаковая глубина для всех), либо dict {var:depth}
    prediction_horizon: int, горизонт предсказания для Dst (лаг в будущее на N часов)
    
    Возвращает датафрейм с исходными признаками и новыми колонками:
    - лаг вперед: Dst_plusN (где N - горизонт предсказания)
    - лаги назад: <var>_lag1 ... <var>_lagN 
    БЕЗ ПРОПУСКОВ
    """
    df = df.copy()    
    missing = [v for v in vars_list if v not in df.columns]
    if missing:
        raise ValueError(f"В df отсутствуют колонки: {missing}")

    if isinstance(embed_map_or_int, int):
        embed_map = {v: embed_map_or_int for v in vars_list}
    else:
        embed_map = embed_map_or_int.copy()
        for v in vars_list:
            if v not in embed_map:
                raise ValueError(f"Не задана глубина для переменной {v} в embed_map")
    
    X = pd.DataFrame(index=df.index)
    
    # Лаги вперед на заданный горизонт предсказания для Dst
    if 'Dst' in df.columns:
        for h in range(1, prediction_horizon+1):
            X[f"Dst_plus{h}"] = df['Dst'].shift(-h)

    print(X)
    
    # Лаги назад для всех переменных из vars_list
    for var in vars_list:
        depth = int(embed_map[var])
        for lag in range(1, depth+1):
            X[f"{var}_lag{lag}"] = df[var].shift(lag)
    
    result = pd.concat([df, X], axis=1)
    result = result.dropna()
    
    return result

In [5]:
# data_ace_to_2021 = pd.read_csv("../data/Compare_ACE_DSCOVR.csv", sep=';', na_values='N', decimal=',')

In [6]:
# В этом файле лежат уже преобработанные и погруженные только на 12 часов данные с аппарата ACE
# data_ace_pogr24_to_2024 = pd.read_csv("../data/Min_browse_data_с_погружением_19971022_20240110_интерполяция12.csv", sep=',', na_values='N', decimal=',')

In [19]:
data_ace_to_2026 = pd.read_csv("../data/All_browse_data_без_погружения_19971021_20260112_с_пропусками.csv", sep=',', na_values='N', decimal='.', encoding='cp1251')

In [20]:
data_discover_to_2026 = pd.read_csv("../data/Discover_с_интреполяцией12_актуальные_данные_до_12.01.2026.csv", sep=';', decimal=',')

In [21]:
data_ace_to_2026

,year,month,day,hour from,hour to,doySin,hourSin,doyCos,hourCos,Dst,...,lg(E>2 MeV) G13,E>2 MeV G15,lg(E>2 MeV) G15,E>2 MeV G14,lg(E>2 MeV) G14,CH_rca (193),CH_rca (211),SW_spd_frcst (193),SW_spd_frcst (211),Unnamed: 78
0,1997,10,21,0,1,-0.505271,0.991445,-0.862961,-0.130526,-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997,10,21,1,2,-0.505889,0.923880,-0.862598,-0.382683,-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1997,10,21,2,3,-0.506508,0.793353,-0.862235,-0.608761,-15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1997,10,21,3,4,-0.507126,0.608761,-0.861872,-0.793353,-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1997,10,21,4,5,-0.507744,0.382683,-0.861508,-0.923880,-13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247459,2026,1,12,19,20,-0.920552,0.382683,0.390621,0.923880,-16,...,NaN,NaN,NaN,NaN,NaN,6.14,10.99,520.26155,488.54081,NaN
247460,2026,1,12,20,21,-0.920271,0.608761,0.391281,0.793353,-18,...,NaN,NaN,NaN,NaN,NaN,6.09,7.77,518.37351,487.35326,NaN
247461,2026,1,12,21,22,-0.919990,0.793353,0.391941,0.608761,-16,...,NaN,NaN,NaN,NaN,NaN,6.46,11.25,517.42268,483.29785,NaN
247462,2026,1,12,22,23,-0.919709,0.923880,0.392601,0.382683,-21,...,NaN,NaN,NaN,NaN,NaN,6.57,7.7,517.42268,479.24244,NaN


In [24]:
data_discover_to_2026

,Date time,Date,hour,bt,bx_gse,by_gse,bz_gse,theta_gse,bx_gsm,by_gsm,...,proton_temperature,alpha_vx_gse,alpha_vy_gse,alpha_vz_gse,alpha_vx_gsm,alpha_vy_gsm,alpha_vz_gsm,alpha_speed,alpha_density,alpha_temperature
0,2016-07-27 00:00:00,2016-07-27,0,2.9384999999999994,1.9816666666666656,0.5636666666666669,0.9568333333333333,20.503999999999998,1.9816666666666656,0.6455000000000002,...,281731.01666666666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2016-07-27 01:00:00,2016-07-27,1,3.175333333333333,2.398166666666666,-1.2258333333333333,1.4093333333333338,27.5085,2.398166666666666,-1.0713333333333332,...,274076.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2016-07-27 02:00:00,2016-07-27,2,3.105666666666666,1.5399999999999996,-2.313833333333333,-0.34816666666666657,-7.204166666666663,1.5399999999999996,-2.337666666666667,...,268199.8333333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-07-27 03:00:00,2016-07-27,3,3.4028333333333323,3.137166666666668,-1.0878333333333332,0.3658333333333331,6.318666666666667,3.137166666666668,-1.0055,...,243006.66666666666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2016-07-27 04:00:00,2016-07-27,4,3.933214285714285,2.3160714285714286,-2.8392857142857153,0.7698214285714285,11.165535714285712,2.3160714285714286,-2.595357142857143,...,242208.30357142858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82819,2026-01-12 19:00:00.494,2026-01-12 19:00:00.494,19,6.54,1.82,-6.23,-0.54,-4.76,1.82,-5.92,...,2000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82820,2026-01-12 20:00:00.494,2026-01-12 20:00:00.494,20,7.27,3.08,-6.54,0.62,4.92,3.08,-6.46,...,2000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82821,2026-01-12 21:00:00.495,2026-01-12 21:00:00.495,21,7.36,5.22,-3.45,-3.86,-31.66,5.22,-2.12,...,163851.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82822,2026-01-12 22:00:00.495,2026-01-12 22:00:00.495,22,7.25,4.52,-2.83,-4.71,-41.45,4.52,-1.16,...,2000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [25]:
# Все необхожимые для обучения моделей признаки
column_names_ace = ['year', 'month', 'day',	'hour from', 'Dst', 'B_x', 'B_gsm_y', 'B_gsm_z', 'B_magn', 'H_den_SWP', 'SW_spd', 'Trr_SWP']
column_names_discover = ['Date time', 'bx_gsm', 'by_gsm', 'bz_gsm', 'bt', 'proton_density', 'proton_speed', 'proton_temperature']

# Создаем словарь для переименования переменных - сделано для того, чтобы было удобно работать с данными обоих аппаратов 
rename_dict = {
    'B_x': 'bx_gsm',
    'B_gsm_y': 'by_gsm', 
    'B_gsm_z': 'bz_gsm',
    'B_magn': 'bt',
    'H_den_SWP': 'proton_density',
    'SW_spd': 'proton_speed',
    'Trr_SWP': 'proton_temperature'
}

# Погружаемые признаки
vars_values = ['Dst', 'bx_gsm', 'by_gsm', 'bz_gsm', 'bt', 'proton_density', 'proton_speed', 'proton_temperature']

# Заданные по автокорреляционной функции глубины погружения
delays_values = [43, 26, 12, 3, 19, 16, 56, 24]
delays = dict(zip(vars_values, delays_values))

# Для скорости СВ вместо всей предыстории в 55 часов оставим только те значения задержки, которые совпадают с числами Фибоначчи (из статьи)
proton_speed_to_drop = [f'proton_speed_lag{i}' for i in range(1, 57) if i not in [1, 2, 3, 5, 8, 13, 21, 34, 55]]

In [69]:
# 0. Копирование загруженных данных в отдельные переменные
data_ace_copy = data_ace_to_2026.copy()
data_discover_copy = data_discover_to_2026.copy()

# 1. Выбор только необходимых признаков для обучения
data_ace_raw = data_ace_copy[column_names_ace]
data_discover_raw = data_discover_copy[column_names_discover]

# 2. Переименование столбцов в ACE
data_ace_raw = data_ace_raw.rename(columns=rename_dict)

# 3. Преобразование столбца с датой и временем
data_ace_raw = set_datetime_index_ace(data_ace_raw)
data_discover_raw = set_datetime_index_discover(data_discover_raw)

# 4. Добавление к данным Discover столбца c dst в соответсвующие даты и время
data_discover_raw = data_ace_raw['Dst'].to_frame().merge(data_discover_raw, how='right', on='datetime')

# 5. Добавление циклической переменной
data_ace_raw = set_cos_sin(data_ace_raw)
data_discover_raw = set_cos_sin(data_discover_raw)

# 6. Осуществление погружения временных рядов на 24 часа по всем переменным с горизонтом предсказания 24 часа
data_ace_24 = create_lagged_features(data_ace_raw, vars_values, 24, prediction_horizon=24)
data_discover_24 = create_lagged_features(data_discover_raw, vars_values, 24, prediction_horizon=24)

# 7. Осуществление погружения временных рядов на глубины, подобранные с учетом корреляционных функций с горизонтом предсказания 24 часа
data_ace_af = create_lagged_features(data_ace_raw, vars_values, delays, prediction_horizon=24)
data_discover_af = create_lagged_features(data_discover_raw, vars_values, delays, prediction_horizon=24)

# 8. Убираем необязательные значения погружения скорости солнечного ветра
data_ace_af = data_ace_af.drop(proton_speed_to_drop, axis=1)
data_discover_af = data_discover_af.drop(proton_speed_to_drop, axis=1)

                     Dst_plus1  Dst_plus2  Dst_plus3  Dst_plus4  Dst_plus5  \
datetime                                                                     
1997-10-21 00:00:00      -15.0      -15.0      -13.0      -13.0      -14.0   
1997-10-21 01:00:00      -15.0      -13.0      -13.0      -14.0      -15.0   
1997-10-21 02:00:00      -13.0      -13.0      -14.0      -15.0      -15.0   
1997-10-21 03:00:00      -13.0      -14.0      -15.0      -15.0      -15.0   
1997-10-21 04:00:00      -14.0      -15.0      -15.0      -15.0      -13.0   
...                        ...        ...        ...        ...        ...   
2026-01-12 19:00:00      -18.0      -16.0      -21.0      -18.0        NaN   
2026-01-12 20:00:00      -16.0      -21.0      -18.0        NaN        NaN   
2026-01-12 21:00:00      -21.0      -18.0        NaN        NaN        NaN   
2026-01-12 22:00:00      -18.0        NaN        NaN        NaN        NaN   
2026-01-12 23:00:00        NaN        NaN        NaN        NaN 

In [70]:
data_ace_24.to_csv("../data/Ace_погружение_24часа.csv", sep=';', encoding='utf-8')
data_discover_24.to_csv("../data/Discover_погружение_24часа.csv", sep=';', encoding='utf-8')
data_ace_af.to_csv("../data/Ace_погружение_AF.csv", sep=';', encoding='utf-8')
data_discover_af.to_csv("../data/Discover_погружение_AF.csv", sep=';', encoding='utf-8')